In [ ]:
# CELL 1 — Environment Check & GPU Verification
import torch, sys, random, os
import numpy as np

print("=" * 60)
print("CELL 1: ENVIRONMENT CHECK & GPU VERIFICATION")
print("=" * 60)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name} | VRAM: {gpu_mem:.1f} GB')
    print(f'CUDA: {torch.version.cuda}')
else:
    print('WARNING: No GPU! Enable in Runtime > Change runtime type > GPU')

print(f'PyTorch: {torch.__version__} | Python: {sys.version.split()[0]} | Device: {device}')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ['PYTHONHASHSEED'] = str(SEED)

# Project Constants
NUM_CLASSES = 62
IMG_SIZE = 128
BATCH_SIZE = 128  # T4-optimized; fallback to 64 if OOM
NUM_EPOCHS = 25
LR = 1e-3
PATIENCE = 5
NUM_WORKERS = 2
CHECKPOINT_PATH = '/kaggle/working/best_model.pth'

CLASS_LABELS = (
    [str(i) for i in range(10)] +
    [chr(i) for i in range(ord('A'), ord('Z')+1)] +
    [chr(i) for i in range(ord('a'), ord('z')+1)]
)
assert len(CLASS_LABELS) == NUM_CLASSES

print(f'\nConfig: {NUM_CLASSES} classes | {IMG_SIZE}x{IMG_SIZE} | batch={BATCH_SIZE} | lr={LR}')
print(f'Seed: {SEED} | Checkpoint: {CHECKPOINT_PATH}')
print("=" * 60)


In [ ]:
# CELL 2 — Dataset Download & Verification
# Option A: Kaggle CLI (run these as shell commands with ! prefix in notebook)
# !pip install kaggle -q
# !kaggle datasets download -d crawford/emnist -p ./data --force
# !unzip -q ./data/emnist.zip -d ./data/emnist_extracted

# Option B: torchvision (recommended - handles everything)
from torchvision.datasets import EMNIST
import os

DATA_ROOT = './data'
os.makedirs(DATA_ROOT, exist_ok=True)

print('Downloading EMNIST ByClass via torchvision...')
train_raw = EMNIST(root=DATA_ROOT, split='byclass', train=True, download=True)
test_raw = EMNIST(root=DATA_ROOT, split='byclass', train=False, download=True)

print(f'\nTraining samples: {len(train_raw):,}')
print(f'Test samples:     {len(test_raw):,}')
print(f'Total:            {len(train_raw) + len(test_raw):,}')
print(f'Classes:          {len(train_raw.classes)}')
print(f'Image shape:      {train_raw[0][0].size} (PIL)')

# Verify folder structure
for root, dirs, files in os.walk(DATA_ROOT):
    level = root.replace(DATA_ROOT, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:
        for f in files[:5]:
            print(f'{indent}  {f}')
        if len(files) > 5:
            print(f'{indent}  ... and {len(files)-5} more files')


In [ ]:
# CELL 3 — Data Exploration (EDA)
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
from torchvision.datasets import EMNIST
from PIL import Image
# from constants import CLASS_LABELS, NUM_CLASSES

train_raw = EMNIST(root='./data', split='byclass', train=True, download=False)

# --- Class Distribution ---
labels = train_raw.targets.numpy()
counter = Counter(labels)

fig, ax = plt.subplots(figsize=(18, 5))
classes_sorted = sorted(counter.keys())
counts = [counter[c] for c in classes_sorted]
colors = ['#2196F3' if c < 10 else '#4CAF50' if c < 36 else '#FF9800' for c in classes_sorted]
ax.bar(range(len(classes_sorted)), counts, color=colors)
ax.set_xticks(range(len(classes_sorted)))
ax.set_xticklabels(CLASS_LABELS, fontsize=7)
ax.set_xlabel('Class')
ax.set_ylabel('Count')
ax.set_title('EMNIST ByClass Distribution (Blue=Digits, Green=Upper, Orange=Lower)')
plt.tight_layout()
plt.savefig('/kaggle/working/class_distribution.png', dpi=150)
plt.show()

print(f'Min samples/class: {min(counts):,} (class {CLASS_LABELS[classes_sorted[np.argmin(counts)]]})')
print(f'Max samples/class: {max(counts):,} (class {CLASS_LABELS[classes_sorted[np.argmax(counts)]]})')
print(f'Imbalance ratio:   {max(counts)/min(counts):.1f}x')

# --- Sample Grid (5x5 per category) ---
fig, axes = plt.subplots(3, 10, figsize=(20, 7))
fig.suptitle('Sample Images (Row1=Digits, Row2=Upper, Row3=Lower)', fontsize=14)
for row, start_class in enumerate([0, 10, 36]):
    for col in range(10):
        cls = start_class + col
        if cls >= NUM_CLASSES:
            axes[row, col].axis('off')
            continue
        idx = (labels == cls).nonzero()[0][0]
        img = train_raw[idx][0]
        # Fix EMNIST orientation: transpose
        img = img.transpose(Image.Transpose.TRANSPOSE)
        axes[row, col].imshow(img, cmap='gray')
        axes[row, col].set_title(CLASS_LABELS[cls], fontsize=10)
        axes[row, col].axis('off')
plt.tight_layout()
plt.savefig('/kaggle/working/sample_grid.png', dpi=150)
plt.show()

# --- Pixel Statistics (for normalization) ---
all_pixels = train_raw.data.float() / 255.0
PIXEL_MEAN = all_pixels.mean().item()
PIXEL_STD = all_pixels.std().item()
print(f'\nPixel Mean: {PIXEL_MEAN:.4f}')
print(f'Pixel Std:  {PIXEL_STD:.4f}')
print('(Use these for transforms.Normalize)')

# --- Check for anomalies ---
zero_images = (train_raw.data.sum(dim=(1, 2)) == 0).sum().item()
print(f'\nBlank/corrupt images: {zero_images}')


In [ ]:
# CELL 4 — Preprocessing Pipeline
# pyrefly: ignore [missing-import]
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, random_split
from torchvision import transforms
from torchvision.datasets import EMNIST
from PIL import Image
import numpy as np
from collections import Counter
# from constants import IMG_SIZE, SEED, BATCH_SIZE, NUM_WORKERS

# --- Custom Dataset with orientation fix + edge case handling ---
class EMNISTByClassDataset(Dataset):
    """Wraps torchvision EMNIST with orientation fix and augmentation."""

    def __init__(self, split='train', transform=None):
        self.data = EMNIST(root='./data', split='byclass', train=(split != 'test'), download=False)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img, label = self.data[idx]

        # Edge case: Fix EMNIST orientation (images are transposed in source)
        img = img.transpose(Image.Transpose.TRANSPOSE)

        # Edge case: Ensure grayscale (handle RGBA/RGB if present)
        if img.mode != 'L':
            img = img.convert('L')

        if self.transform:
            img = self.transform(img)

        return img, label

# --- Transforms ---
# Use PIXEL_MEAN and PIXEL_STD from Cell 3 (typical: ~0.1736, ~0.3317)
PIXEL_MEAN = 0.1736
PIXEL_STD = 0.3317

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1), shear=5),
    transforms.ElasticTransform(alpha=50.0, sigma=5.0),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 0.5)),
    transforms.ToTensor(),
    transforms.Normalize([PIXEL_MEAN], [PIXEL_STD]),
    transforms.RandomErasing(p=0.1),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize([PIXEL_MEAN], [PIXEL_STD]),
])

# --- Create Datasets ---
full_train = EMNISTByClassDataset(split='train', transform=train_transform)
test_dataset = EMNISTByClassDataset(split='test', transform=val_transform)

# 80/10/10 split from training data (test set is already separate)
total = len(full_train)
val_size = int(0.1 * total)
train_size = total - val_size

train_dataset, val_dataset = random_split(
    full_train, [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)
# Override val transforms (random_split keeps parent transforms)
val_dataset_clean = EMNISTByClassDataset(split='train', transform=val_transform)
val_indices = val_dataset.indices
val_dataset = torch.utils.data.Subset(val_dataset_clean, val_indices)

print(f'Train: {train_size:,} | Val: {val_size:,} | Test: {len(test_dataset):,}')

# --- Edge Case: Class Imbalance → WeightedRandomSampler ---
train_labels = [full_train.data.targets[i].item() for i in train_dataset.indices]
class_counts = Counter(train_labels)
class_weights = {c: 1.0 / count for c, count in class_counts.items()}
sample_weights = [class_weights[l] for l in train_labels]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

# --- DataLoaders ---
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

print(f'Batches per epoch: {len(train_loader):,}')
print(f'Batch size: {BATCH_SIZE} | Workers: {NUM_WORKERS}')

# Quick sanity check
batch_imgs, batch_labels = next(iter(train_loader))
print(f'Batch shape: {batch_imgs.shape} | Labels shape: {batch_labels.shape}')
print(f'Pixel range: [{batch_imgs.min():.2f}, {batch_imgs.max():.2f}]')


In [ ]:
# CELL 5 — Model Setup (EfficientNet-B0 Fine-Tuning)
import torch
import torch.nn as nn
import torchvision.models as models
# from constants import NUM_CLASSES, IMG_SIZE, device

# --- Decision Logic ---
DATASET_SIZE = len(full_train)
USE_PRETRAINED = True  # Fine-tune even with >200k images; transfer learning still helps

print(f'Dataset size: {DATASET_SIZE:,} images')
if DATASET_SIZE > 200_000:
    print('Dataset > 200k: Could train from scratch, but fine-tuning is STILL faster + better.')
print(f'Strategy: {"Fine-tune EfficientNet-B0 (ImageNet)" if USE_PRETRAINED else "Train from scratch"}')

# --- Option B: Fine-tune EfficientNet-B0 (WINNER) ---
def build_model(num_classes=NUM_CLASSES, pretrained=True):
    if pretrained:
        weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1
        model = models.efficientnet_b0(weights=weights)
    else:
        model = models.efficientnet_b0(weights=None)

    # Modify input: 3ch RGB → 1ch grayscale
    original_conv = model.features[0][0]
    model.features[0][0] = nn.Conv2d(
        1, 32, kernel_size=3, stride=2, padding=1, bias=False
    )
    # Initialize from pretrained: average RGB weights into single channel
    if pretrained:
        with torch.no_grad():
            model.features[0][0].weight = nn.Parameter(
                original_conv.weight.mean(dim=1, keepdim=True)
            )

    # Modify output: 1000 ImageNet classes → 62 EMNIST classes
    in_features = model.classifier[1].in_features  # 1280
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(in_features, num_classes)
    )

    # --- Freeze/Unfreeze Strategy ---
    # Freeze all backbone layers first
    for param in model.features.parameters():
        param.requires_grad = False

    # Unfreeze last 2 conv blocks (features[7] and features[8]) for fine-tuning
    for param in model.features[7].parameters():
        param.requires_grad = True
    for param in model.features[8].parameters():
        param.requires_grad = True

    # Classifier is always trainable
    for param in model.classifier.parameters():
        param.requires_grad = True

    return model

# --- Option A: Train from scratch (alternative) ---
class CharCNN(nn.Module):
    """Lightweight CNN for training from scratch if needed."""
    def __init__(self, num_classes=62):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.AdaptiveAvgPool2d(4),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.4),
            nn.Linear(256 * 4 * 4, 512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.classifier(self.features(x))

# --- Build chosen model ---
model = build_model(num_classes=NUM_CLASSES, pretrained=USE_PRETRAINED)
model = model.to(device)

# Parameter count
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f'\nModel: EfficientNet-B0')
print(f'Total params:     {total_params:,}')
print(f'Trainable params: {trainable_params:,}')
print(f'Frozen params:    {frozen_params:,}')
print(f'Input:  1x{IMG_SIZE}x{IMG_SIZE} grayscale')
print(f'Output: {NUM_CLASSES} classes')


In [ ]:
# CELL 6 — Production Training Loop
import time
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
# from constants import (
    LR, NUM_EPOCHS, PATIENCE, device,
    CLASS_LABELS, NUM_CLASSES, IMG_SIZE,
    PIXEL_MEAN, PIXEL_STD, CHECKPOINT_PATH
)

# --- Optimizer & Scheduler ---
optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=1e-4)
scheduler = OneCycleLR(optimizer, max_lr=LR, steps_per_epoch=len(train_loader), epochs=NUM_EPOCHS)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler = GradScaler('cuda')

# --- OOM Fallback ---
GRAD_ACCUM_STEPS = 1  # Increase to 2 if OOM with batch_size=128
MAX_GRAD_NORM = 1.0

# --- Early Stopping ---
class EarlyStopping:
    def __init__(self, patience=5):
        self.patience = patience
        self.counter = 0
        self.best_score = None
        self.should_stop = False

    def __call__(self, val_acc):
        if self.best_score is None or val_acc > self.best_score:
            self.best_score = val_acc
            self.counter = 0
            return True  # improved
        self.counter += 1
        if self.counter >= self.patience:
            self.should_stop = True
        return False  # not improved

early_stop = EarlyStopping(patience=PATIENCE)
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

# --- Training Loop ---
print(f'Training on {device} | Epochs: {NUM_EPOCHS} | Patience: {PATIENCE}')
print(f'Mixed Precision: ON | Grad Clipping: {MAX_GRAD_NORM} | Accum Steps: {GRAD_ACCUM_STEPS}')
print('=' * 70)

for epoch in range(NUM_EPOCHS):
    t0 = time.time()

    # --- Train ---
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0
    optimizer.zero_grad()

    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)

        try:
            with autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels) / GRAD_ACCUM_STEPS

            scaler.scale(loss).backward()

            if (batch_idx + 1) % GRAD_ACCUM_STEPS == 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                scheduler.step()

        except RuntimeError as e:
            if 'out of memory' in str(e):
                print(f'OOM at batch {batch_idx}! Try reducing BATCH_SIZE to 64 with GRAD_ACCUM_STEPS=2')
                torch.cuda.empty_cache()
                continue
            raise e

        train_loss += loss.item() * GRAD_ACCUM_STEPS * images.size(0)
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_total += labels.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total

    # --- Validate ---
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            with autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total

    # Log
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    elapsed = time.time() - t0
    improved = early_stop(val_acc)

    marker = ' *** BEST ***' if improved else ''
    print(f'Epoch {epoch+1:02d}/{NUM_EPOCHS} | '
          f'Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | '
          f'Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | '
          f'{elapsed:.0f}s{marker}')

    # Save best checkpoint
    if improved:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'val_loss': val_loss,
            'class_labels': CLASS_LABELS,
            'config': {'num_classes': NUM_CLASSES, 'img_size': IMG_SIZE,
                       'pixel_mean': PIXEL_MEAN, 'pixel_std': PIXEL_STD}
        }, CHECKPOINT_PATH)

    if early_stop.should_stop:
        print(f'\nEarly stopping at epoch {epoch+1}. Best val_acc: {early_stop.best_score:.4f}')
        break

print('=' * 70)
print(f'Training complete. Best val_acc: {early_stop.best_score:.4f}')

# --- Plot Training Curves ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history['train_loss'], label='Train'); ax1.plot(history['val_loss'], label='Val')
ax1.set_title('Loss'); ax1.legend(); ax1.set_xlabel('Epoch')
ax2.plot(history['train_acc'], label='Train'); ax2.plot(history['val_acc'], label='Val')
ax2.set_title('Accuracy'); ax2.legend(); ax2.set_xlabel('Epoch')
plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=150)
plt.show()


In [ ]:
# CELL 7 — Evaluation
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.amp import autocast
# from constants import CLASS_LABELS, NUM_CLASSES, device, CHECKPOINT_PATH

# Load best checkpoint
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
print(f'Loaded best model from epoch {checkpoint["epoch"]+1} (val_acc={checkpoint["val_acc"]:.4f})')

# --- Run on test set ---
model.eval()
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device, non_blocking=True)
        with autocast('cuda'):
            outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        preds = outputs.argmax(1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

test_acc = (all_preds == all_labels).mean()
print(f'\nTest Accuracy: {test_acc:.4f} ({(all_preds == all_labels).sum():,}/{len(all_labels):,})')

# --- Confusion Matrix ---
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(18, 16))
sns.heatmap(cm, annot=False, fmt='d', cmap='Blues', xticklabels=CLASS_LABELS,
            yticklabels=CLASS_LABELS, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True', fontsize=12)
ax.set_title(f'Confusion Matrix (Test Acc: {test_acc:.4f})', fontsize=14)
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix.png', dpi=150)
plt.show()

# --- Per-Class Report ---
report = classification_report(all_labels, all_preds, target_names=CLASS_LABELS, output_dict=True)
print('\n--- Per-Class Precision / Recall / F1 ---')
print(classification_report(all_labels, all_preds, target_names=CLASS_LABELS))

# --- Top 5 Most Confused Pairs ---
np.fill_diagonal(cm, 0)
confused_pairs = []
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if i != j and cm[i, j] > 0:
            confused_pairs.append((cm[i, j], CLASS_LABELS[i], CLASS_LABELS[j]))
confused_pairs.sort(reverse=True)

print('\n--- Top 5 Most Confused Pairs ---')
for count, true_cls, pred_cls in confused_pairs[:5]:
    print(f'  {true_cls} → {pred_cls}: {count:,} misclassifications')

# --- Visual Grid of Misclassified Samples ---
misclassified_idx = np.where(all_preds != all_labels)[0]
fig, axes = plt.subplots(2, 5, figsize=(15, 7))
fig.suptitle('Misclassified Samples (True → Predicted)', fontsize=14)
for i, ax in enumerate(axes.flat):
    if i >= len(misclassified_idx):
        ax.axis('off')
        continue
    idx = misclassified_idx[i]
    img, _ = test_dataset[idx]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(f'{CLASS_LABELS[all_labels[idx]]}→{CLASS_LABELS[all_preds[idx]]}',
                 color='red', fontsize=11)
    ax.axis('off')
plt.tight_layout()
plt.savefig('/kaggle/working/misclassified.png', dpi=150)
plt.show()


In [ ]:
# CELL 8 — Inference Pipeline
import torch
import torch.nn.functional as F
from torch.amp import autocast
from torchvision import transforms
from PIL import Image
import numpy as np

def load_model_for_inference(checkpoint_path, device='cuda'):
    """Load trained model from checkpoint."""
    checkpoint = torch.load(checkpoint_path, map_location=device)
    config = checkpoint['config']

    model = build_model(num_classes=config['num_classes'], pretrained=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    model.eval()

    preprocess = transforms.Compose([
        transforms.Resize((config['img_size'], config['img_size'])),
        transforms.ToTensor(),
        transforms.Normalize([config['pixel_mean']], [config['pixel_std']]),
    ])
    return model, preprocess, checkpoint['class_labels']

def predict_single_image(img_path_or_pil, model, preprocess, class_labels, device='cuda'):
    """
    Predict a single image. Accepts file path or PIL Image.
    Returns: predicted class, confidence, top-3 predictions.
    """
    # Load image
    if isinstance(img_path_or_pil, str):
        img = Image.open(img_path_or_pil)
    else:
        img = img_path_or_pil

    # Edge case: convert to grayscale
    if img.mode != 'L':
        img = img.convert('L')

    # Edge case: aspect-ratio-preserving resize with padding
    w, h = img.size
    max_dim = max(w, h)
    padded = Image.new('L', (max_dim, max_dim), 0)
    padded.paste(img, ((max_dim - w) // 2, (max_dim - h) // 2))
    img = padded

    # Preprocess and predict
    tensor = preprocess(img).unsqueeze(0).to(device)

    with torch.no_grad():
        with autocast('cuda'):
            logits = model(tensor)
    probs = F.softmax(logits, dim=1).cpu().squeeze()

    # Top-3 predictions
    top3_probs, top3_idx = probs.topk(3)
    top3 = [(class_labels[idx], prob.item()) for idx, prob in zip(top3_idx, top3_probs)]

    predicted_class = top3[0][0]
    confidence = top3[0][1]

    return predicted_class, confidence, top3

# --- Demo ---
inf_model, inf_preprocess, inf_labels = load_model_for_inference(CHECKPOINT_PATH, device)

# Test on a random test image
demo_img, demo_label = test_dataset[42]
demo_pil = transforms.ToPILImage()(demo_img)

pred_class, conf, top3 = predict_single_image(demo_pil, inf_model, inf_preprocess, inf_labels, device)

print(f'True Label:  {CLASS_LABELS[demo_label]}')
print(f'Predicted:   {pred_class} (confidence: {conf:.4f})')
print(f'Top-3:')
for cls, prob in top3:
    print(f'  {cls}: {prob:.4f}')


In [ ]:
# CELL 9 — Export & Save
import torch
import os

# --- Save as .pth ---
print('1. Saving model as .pth ...')
final_save_path = '/kaggle/working/emnist_efficientnet_b0_62cls.pth'
torch.save({
    'model_state_dict': model.state_dict(),
    'class_labels': CLASS_LABELS,
    'num_classes': NUM_CLASSES,
    'architecture': 'efficientnet_b0',
    'img_size': IMG_SIZE,
    'pixel_mean': PIXEL_MEAN,
    'pixel_std': PIXEL_STD,
    'best_val_acc': early_stop.best_score,
}, final_save_path)
print(f'   Saved: {final_save_path} ({os.path.getsize(final_save_path)/1e6:.1f} MB)')

# --- Export to ONNX ---
print('\n2. Exporting to ONNX ...')
onnx_path = '/kaggle/working/emnist_efficientnet_b0_62cls.onnx'
model.eval()
model_cpu = model.cpu()
dummy_input = torch.randn(1, 1, IMG_SIZE, IMG_SIZE)

torch.onnx.export(
    model_cpu, dummy_input, onnx_path,
    input_names=['image'],
    output_names=['logits'],
    dynamic_axes={'image': {0: 'batch'}, 'logits': {0: 'batch'}},
    opset_version=17
)
print(f'   Saved: {onnx_path} ({os.path.getsize(onnx_path)/1e6:.1f} MB)')

# Move model back to GPU
model = model.to(device)

# --- Verify ONNX ---
try:
    import onnx
    onnx_model = onnx.load(onnx_path)
    onnx.checker.check_model(onnx_model)
    print('   ONNX model verified OK')
except ImportError:
    print('   onnx package not installed; skip verification (pip install onnx)')

# --- Upload to Kaggle Models (optional) ---
print('\n3. To upload to Kaggle Models, run:')
print('   !kaggle models instances versions create \\')
print('     --model <your-username>/emnist-efficientnet-b0 \\')
print('     --instance-slug default \\')
print('     --version-notes "EfficientNet-B0 on EMNIST ByClass 62 classes" \\')
print('     --dir-or-file /kaggle/working/')

# --- Final Summary ---
print('\n' + '=' * 60)
print('FINAL OUTPUT FILES:')
print('=' * 60)
for f in ['best_model.pth', 'emnist_efficientnet_b0_62cls.pth',
          'emnist_efficientnet_b0_62cls.onnx',
          'class_distribution.png', 'sample_grid.png',
          'training_curves.png', 'confusion_matrix.png', 'misclassified.png']:
    fpath = f'/kaggle/working/{f}'
    if os.path.exists(fpath):
        size = os.path.getsize(fpath) / 1e6
        print(f'  [OK] {f} ({size:.1f} MB)')
    else:
        print(f'  [--] {f} (not found)')

print('\nDone! Notebook complete.')
